
# COM550 - Processamento de Linguagem Natural

## Semana 4 - *Embeddings*
## Videoaula 12 - Prática: *Embeddings*

Este código está composto de duas partes:

1. Na primeira, fazemos o *download* e a utilização das *word embeddings* pré-treinadas do NILC
2. Na segunda, são criadas nossas próprias *word embeddings* a partir de um *corpus*



# 1. Usando *word embeddings* pré-treinadas
Algumas opções de *word embeddings* pré-treinadas para o português são:

* *Word embeddings* monolíngues (ex: NILC *word embeddings* disponíveis em http://nilc.icmc.usp.br/embeddings))
* *Word embeddings* multilíngues (ex: MUSE *word embedings* disponíveis em https://github.com/facebookresearch/MUSE)

Foram escolhidas as *word embeddings* do NILC treinadas com GloVe conforme informações disponíveis na página das NILC *word embeddings* citada acima.

In [ ]:
!pip install torch pot gensim transformers

In [ ]:
import gensim

In [ ]:
#!wget http://143.107.183.175:22980/download.php?file=embeddings/glove/glove_s300.zip -O glove_s300.zip

In [ ]:
#!unzip -o 'glove_s300.zip'

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file

path = hf_hub_download(repo_id="nilc-nlp/glove-300d",
                       filename="embeddings.safetensors")

data = load_file(path)
vectors = data["embeddings"]

vocab_path = hf_hub_download(repo_id="nilc-nlp/glove-300d",
                             filename="vocab.txt")
with open(vocab_path) as f:
    vocab = [w.strip() for w in f]

print(vectors.shape)

(929605, 300)


In [ ]:
# Inicializa um modelo KeyedVectors vazio com o tamanho correto dos vetores
nilc_glove = gensim.models.KeyedVectors(vector_size=vectors.shape[1])

# Adiciona as palavras (vocab) e seus vetores ao modelo.
# Assume-se que a ordem em `vocab` corresponde à ordem dos vetores em `vectors`.
nilc_glove.add_vectors(keys=vocab, weights=vectors)

In [ ]:
nilc_glove['misógino']

array([-2.82590e-01, -2.47344e-01, -3.10648e-01, -5.33403e-01,
       -8.74540e-02, -2.86342e-01,  2.23928e-01,  1.93496e-01,
        1.56978e-01, -2.06220e-01,  2.56833e-01,  2.30822e-01,
       -2.44496e-01, -3.46459e-01, -2.14285e-01,  1.28289e-01,
        5.00071e-01, -1.16759e-01,  3.74183e-01,  5.70473e-01,
        3.41010e-01,  6.97590e-02,  3.28582e-01, -4.96584e-01,
       -2.53830e-02,  2.93280e-02,  3.89171e-01, -1.00390e-01,
        2.22110e-02, -2.66824e-01,  1.10144e-01,  1.64915e-01,
        4.66531e-01, -2.36450e-01,  4.39500e-02, -2.31796e-01,
        8.32450e-02, -6.10374e-01,  4.99260e-01, -2.62471e-01,
        4.06617e-01,  1.73556e-01,  2.40664e-01, -3.28529e-01,
       -9.95245e-01,  9.87780e-02, -4.18667e-01, -5.30869e-01,
       -3.14080e-01, -1.04422e-01,  3.82100e-01, -1.58487e-01,
       -3.89960e-02, -3.07174e-01,  5.53560e-02, -2.09350e-01,
       -1.53416e-01, -7.08290e-01,  7.70400e-03,  4.58489e-01,
       -8.38238e-01, -8.83700e-02,  7.85040e-02, -1.246

In [ ]:
resultado = nilc_glove.most_similar(positive=['amar', 'odiando'], negative=['odiar'])
print("{}: {:.4f}".format(*resultado[0]))

amando: 0.3852


Também podemos conferir quais são as *n* palavras mais próximas (o *default* do most_similar é *n* = 10) de uma determinada palavra, por exemplo: "escola".

In [ ]:
nilc_glove.most_similar("mulher")

[('mãe', 0.7325751185417175),
 ('esposa', 0.7167492508888245),
 ('marido', 0.7122547030448914),
 ('filha', 0.69303297996521),
 ('menina', 0.6832178235054016),
 ('ela', 0.6737655401229858),
 ('homem', 0.6715222597122192),
 ('criança', 0.6485750079154968),
 ('irmã', 0.6453099846839905),
 ('jovem', 0.6391358971595764)]

Também é possível calcular o valor de similaridade (usando cosseno, por *default*) entre duas palavras.

In [ ]:
print("Similaridade entre menino e menina: ", nilc_glove.similarity("menino","menina"))

Similaridade entre menino e menina:  0.7013513


E verificar qual palavra distoa em um grupo de quatro palavras, por exemplo: menina, menino, cão e garota.

In [ ]:
print(nilc_glove.doesnt_match("menina menino cão garota".split()))

cão


# Verificando a similaridade entre sentenças

Além de verificar a similaridade entre palavas, o gensim disponibliza algumas rotinas para calcular a similaridade entre sentenças. Por exemplo, a similaridade da sentença "O menino foi para a escola de ônibus." e as sentenças "Ana foi para a escola ontem." e "A menina foi para casa.". Veja que a similaridade com a última é maior do que com a primeira, isso as similaridades das palavras é maior.

In [ ]:
sentenca = "O menino foi para a escola de ônibus.".lower().split()

sent1 = "Ana foi para a escola ontem.".lower().split()
similaridade = nilc_glove.wmdistance(sentenca, sent1)
print("{:.4f}".format(similaridade))

sent2 = "A menina foi para casa.".lower().split()
similaridade = nilc_glove.wmdistance(sentenca, sent2)
print("{:.4f}".format(similaridade))

0.5826
0.6676


# BERTimbau

Modelo de linguagem contextualizado para o português.

In [ ]:
import torch
from transformers import AutoTokenizer  # ou BertTokenizer
from transformers import AutoModelForPreTraining  # ou BertForPreTraining para carregar pretraining heads
from transformers import AutoModel  # ou BertModel, para BERT sem pretraining heads

In [ ]:
#device = 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

tokenizer = AutoTokenizer.from_pretrained('neuralmind/bert-large-portuguese-cased', do_lower_case=False)
bert = AutoModel.from_pretrained('neuralmind/bert-large-portuguese-cased')
bert = bert.to(device)

In [ ]:
texto = 'Eu vou ao banco pagar a conta hoje.'

# tokenizando o texto
input_ids = tokenizer.encode(texto, return_tensors='pt')
wordpieces = tokenizer.convert_ids_to_tokens(input_ids[0])

# salvando ponteiros para palavras
subwords_idx = [] # first subword of each word
for i, wordpiece in enumerate(wordpieces):
    if '##' not in wordpiece and i not in [0, len(wordpieces)-1]:
        subwords_idx.append(i)

# obtendo os vetores para as palavras
input_ids = input_ids.to(device)
with torch.no_grad():
  outs = bert(input_ids)
  vetores = outs[0][0, :]

print(vetores[subwords_idx].size(), '\n')
vetores[subwords_idx]

# 2. Gerando suas próprias *word embeddings*

Agora vamos gerar WEs para um *corpus*. Veja que todas as rotinas e os cálculos apresentados anteriormente continuam aplicáveis com as novas WEs que vamos gerar!

Para gerar nossas WEs vamos precisar de um *corpus*...

#O *corpus*

Utilizaremos os dados de um desafio do Kaggle para análise de sentimentos, em português:

* https://www.kaggle.com/augustop/portuguese-tweets-for-sentiment-analysis

In [ ]:
!wget https://www.dropbox.com/s/bdr4tmkxcvk9pqc/NoThemeTweets.csv.zip?dl=1 -O NoThemeTweets.csv.zip
!unzip -o NoThemeTweets.csv.zip

Vamos utilizar o pandas para visualizar os campos do arquivo .csv que baixamos.

In [ ]:
import pandas as pd
corpus = pd.read_csv('NoThemeTweets.csv', sep=',')
corpus.head(10)

Vamos utilizar somente o campo tweet_text, que contém os *tweets*, nosso objeto de interesse. Por exemplo, o terceiro *tweet* é acessado na posição 2 (as posições começam no 0).

In [ ]:
tweets = corpus["tweet_text"].values
tweets[2]

#Processamento básico
Neste processamento básico, utilizaremos a biblioteca NLTK.

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words('portuguese')

O primeiro passo é "ler" e armazenar todas as sentenças (*tweets*) do nosso *corpus*, convertendo as palavras para minúsculas e removendo *stopwords* e *tokens* que não tenham letras nem números.


In [ ]:
sentencas = []
for tweet in tweets:
    tokens = nltk.word_tokenize(tweet)
    processada = [w.lower() for w in tokens if not w in stopwords and w.isalnum()]
    sentencas.append(processada)
len(sentencas)

No total, nosso *corpus* possui `785.814` sentenças. Após o processamento acima, a terceira sentença ficou:

In [ ]:
sentencas[2]

# Gerando *word embeddings*
Agora estamos prontos para gerar nossas WEs a partir dessas sentenças. Para isso, basta executar o código abaixo, que gera WEs usando word2vec (um dos algoritmos mais tradicionais para geração de WEs) com dimensão 300 (a mesma das WEs do NILC).

In [ ]:
from gensim.models import Word2Vec
meu_modelo = Word2Vec(sentencas,
                 min_count=3,     # Ignora palavras que ocorrem menos de 3 vezes
                 vector_size=300, # Dimensão das embeddings
                 workers=2,       # Número de processadores (paralelização)
                 window=5,        # Tamanho da janela de contexto, em palavras, usada durante o treinamento
                 epochs=30)       # Número de épocas de treinamento

É possível conferir a quantidade de palavras para as quais foram geradas WEs:

In [ ]:
len(meu_modelo.wv.index_to_key)

E dar uma olhada em quais são essas palavras:

In [ ]:
print(meu_modelo.wv.key_to_index)

Nessas novas WEs, as palavras mais similares à "escola" são:

In [ ]:
meu_modelo.wv.most_similar("escola")

É interessante notar que o contexto mais usual do Twitter faz com que seja possível encontrar gíras como "facul" e "uni" como palavras similares à "escola".

# Salvando as *word embeddings* geradas

Por fim, podemos salvar o modelo treinado para ser carregado para uso futuro (de modo similar ao que fizemos na parte 1 deste código). É importante salvar o modelo, pois o CoLab vai remover tudo assim que esta seção terminar.

In [ ]:
meu_modelo.wv.save_word2vec_format('meu_modelo.bin', binary=True)

# Visualizando as *word embeddings*
Uma das coisas mais legais para se fazer com as WEs é *plotá-las* em duas dimensões para analisar visualmente a distância entre as palavras. É isso o que faremos a seguir.

In [ ]:
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('classic')
from sklearn.decomposition import PCA

In [ ]:
def display_pca_scatterplot(model, words):
    word_vectors = np.array([model[w] for w in words])

    twodim = PCA().fit_transform(word_vectors)[:,:2]

    plt.figure(figsize=(6,6))
    plt.scatter(twodim[:,0], twodim[:,1], edgecolors='k', c='r')
    for word, (x,y) in zip(words, twodim):
        plt.text(x+0.05, y+0.05, word)

In [ ]:
meu_modelo.wv.most_similar('escola')

*Plotando* o vetor da palavra "escola" e as 10 palavras mais próximas a ele, de acordo com o modelo que acabamos de treinar.

In [ ]:
words = ['escola', 'faculdade', 'facul', 'aula', 'sala', 'colégio', 'academia', 'casa', 'escolinha', 'uni', 'cursinho']
display_pca_scatterplot(meu_modelo.wv,words)

*Plotando* o vetor da palavra "escola" e as 10 palavras mais próximas a ele, de acordo com o modelo do NILC.

In [ ]:
nilc_glove.most_similar('escola')

In [ ]:
words = ['escola', 'alunos', 'escolas', 'ensino', 'colégio', 'faculdade', 'aula', 'aulas', 'educação', 'professores', 'escolar']
display_pca_scatterplot(nilc_glove,words)

#Agradecimentos

Aula baseada no curso da Profa. Dra. Helena Caseli (UFSCar)

Fontes:
* https://radimrehurek.com/gensim/models/keyedvectors.html
* http://nilc.icmc.usp.br/embeddings
* https://github.com/neuralmind-ai/portuguese-bert
